In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
import ipywidgets as widgets
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd

load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)

In [2]:
folder_path = 'data_for_adm03_bangladesh'

shp_files = [f for f in os.listdir(folder_path) if f.endswith('.shp')]

if not shp_files:
    raise FileNotFoundError(f"didn't found any shp files in {folder_path}")

shp_path = os.path.join(folder_path, shp_files[0])
print(f"this is the file {shp_path}")

this is the file data_for_adm03_bangladesh\bgd_admbnda_adm3_bbs_20180410.shp


In [3]:
import geopandas as gpd

#read and covert to geojson
gdf = gpd.read_file(shp_path)
print(f"Loaded {len(gdf)} features")
geojson = gdf.__geo_interface__

python_list = list(zip(gdf['ADM2_EN'], gdf['ADM3_EN']))
print(python_list[:5])

print("total upzillas: ", len(python_list))

Loaded 544 features
[('Jessore', 'Abhaynagar'), ('Dhaka', 'Adabor'), ('Bogra', 'Adamdighi'), ('Lalmonirhat', 'Aditmari'), ('Barisal', 'Agailjhara')]
total upzillas:  544


In [4]:
#just fixing if there is any tiny, unnecessary vertices
gdf['geometry'] = gdf.geometry.simplify(tolerance=0.001, preserve_topology=True)

import numpy as np
np.random.seed(1)
gdf['random_id'] = np.random.rand(len(gdf))
def get_color(feature):
    value = feature['properties']['random_id']
    palette = ['#FF0000', '#00FF00', '#0000FF', '#FFFF00', '#00FFFF', '#FF00FF', '#FFA500']
    idx = int(value * len(palette))
    idx = min(idx, len(palette) - 1)
    return {
        'fillColor': palette[idx],
        'color': 'black', #this is border color
        'weight': 1,     #this is border weight
        'fillOpacity': 0.7
    }

m = geemap.Map()
m.setCenter(90.35, 23.68, 10)
m.add_gdf(
    gdf, 
    layer_name='upzillas', 
    style_callback=get_color,
    info_mode='on_click' #for the inspactor map
)

m


Map(center=[23.68, 90.35], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [6]:
water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select('occurrence')
permanent_water = water.gt(50)
water_area = permanent_water.multiply(ee.Image.pixelArea())


In [7]:
upazilas = geemap.geopandas_to_ee(gdf)
water_stats = water_area.reduceRegions(
    collection=upazilas,
    reducer=ee.Reducer.sum(),
    scale=30,  # 30m resolution
    crs='EPSG:3857'
)

In [8]:
m = geemap.Map()
m.setCenter(90.35, 23.68, 7)
m.set_options('HYBRID')

wettest_upazilas = water_stats.sort('sum', False)

m.addLayer(permanent_water.selfMask(), {'palette': '0000FF'}, 'Permanent Water')


wetness_vis = ee.Image().paint(wettest_upazilas, 'sum')
vis_params = {
    'palette': ['white', 'cyan', 'blue'], 
    'min': 0, 
    'max': 100000000  #100 sqkm
}
m.addLayer(wetness_vis, vis_params, 'water intensity')

# borders = ee.Image().paint(upazilas, 0, 1)
# m.addLayer(borders, {'palette': 'white'}, 'borders')

m.addLayer(upazilas, {'color': '00000000'}, 'Upazila Info (Hidden)', True, 0.5)

In [9]:
m

Map(center=[23.68, 90.35], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…